# 一、前言

今天的主要任务就是手写一份 CNN on CIFAR-10，且目标 test acc > 70%

# 二、代码

## 1.加载数据

In [45]:
import torch
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
# 归一化用 CIFAR-10 官方均值/方差；效果比 [0,1] 归一好
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),      # 随机裁剪，相当于免费送 4px 平移
    transforms.RandomHorizontalFlip(),          # 一半概率水平翻转
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

# root 放到仓库的数据目录，别散在 ~/.cache
train_data = datasets.CIFAR10(root = '/home/wsl2/py-learning-log/21-torch/data', train = True,
                              transform = transform, download = True)
test_data = datasets.CIFAR10(root = '/home/wsl2/py-learning-log/21-torch/data', train = False,
                             transform = transform, download = True)
train_loader = DataLoader(train_data, batch_size = 128, shuffle = True, num_workers = 2)
test_loader = DataLoader(test_data, batch_size = 128, shuffle = False, num_workers = 2)
# 验证 shape：([128, 3, 32, 32]) 和 10 类
X, y = next(iter(train_loader))
print(X.shape, y.shape, len(train_data), len(test_data))

torch.Size([128, 3, 32, 32]) torch.Size([128]) 50000 10000


## 2."用机器推维度"

In [46]:
x = torch.randn(1, 3, 32, 32)
conv1 = torch.nn.Conv2d(3, 16, kernel_size = 3, padding = 1)
x = torch.relu(conv1(x))
print('卷积后:', x.shape)
pool = torch.nn.MaxPool2d(2)
x = pool(x)
print('池化后:', x.shape)
x = x.flatten(1)
print('展平后:', x.shape)

卷积后: torch.Size([1, 16, 32, 32])
池化后: torch.Size([1, 16, 16, 16])
展平后: torch.Size([1, 4096])


## 3.确定网络结构

In [47]:
net = nn.Sequential(
    nn.Conv2d(3, 16, kernel_size = 3, padding = 1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    
    nn.Conv2d(16, 32, kernel_size = 3, padding = 1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    
    nn.Conv2d(32, 64, kernel_size = 3, padding = 1),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Flatten(),
    nn.Linear(64 * 4 * 4, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 10))

net.to(device)

Sequential(
  (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): ReLU()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (7): ReLU()
  (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (9): Flatten(start_dim=1, end_dim=-1)
  (10): Linear(in_features=1024, out_features=512, bias=True)
  (11): ReLU()
  (12): Dropout(p=0.5, inplace=False)
  (13): Linear(in_features=512, out_features=256, bias=True)
  (14): ReLU()
  (15): Dropout(p=0.5, inplace=False)
  (16): Linear(in_features=256, out_features=10, bias=True)
)

## 4.训练

In [49]:
from torch.optim.lr_scheduler import CosineAnnealingLR
loss = nn.CrossEntropyLoss(reduction = 'none')
trainer = torch.optim.SGD(net.parameters(), lr = 0.1, momentum = 0.9, weight_decay = 5e-4)
scheduler = CosineAnnealingLR(trainer, T_max = num_epochs)
num_epochs = 30

def accuracy(y_hat, y):
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        y_hat = y_hat.argmax(axis = 1)      # 每行取概率最大的列号 = 预测类别
    cmp = (y_hat.type(y.dtype) == y)       # 逐元素比较，True/False
    return float(cmp.type(y.dtype).sum())  # 统计预测对的个数

def evaluate_accuracy(net, data_iter, device):
    acc_sum, n = 0.0, 0
    net.eval()
    with torch.no_grad():
        for X, y in data_iter:
            X, y = X.to(device), y.to(device)
            acc_sum += accuracy(net(X),y)     # 每个批次猜对的个数
            n += y.numel()                     # 累计总样本数
    return acc_sum / n                     # 猜对个数 ÷ 总数

for epoch in range(num_epochs):
    train_loss, train_acc, n = 0.0, 0.0, 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        trainer.zero_grad()
        y_hat = net(X)
        l = loss(y_hat, y)
        l.mean().backward()
        trainer.step()
        train_loss += l.sum().item()
        train_acc += accuracy(y_hat, y)
        n += y.numel()
        scheduler.step()
    test_acc = evaluate_accuracy(net, test_loader, device)
    print(f'epoch{epoch + 1}, loss {train_loss / n:.3f} ' 
          f'train acc {train_acc / n:.3f}, test acc {test_acc:.3f}')

epoch1, loss 0.506 train acc 0.830, test acc 0.740
epoch2, loss 0.396 train acc 0.866, test acc 0.723
epoch3, loss 0.364 train acc 0.877, test acc 0.729
epoch4, loss 0.347 train acc 0.882, test acc 0.704
epoch5, loss 0.372 train acc 0.874, test acc 0.732
epoch6, loss 0.327 train acc 0.890, test acc 0.714
epoch7, loss 0.327 train acc 0.889, test acc 0.736
epoch8, loss 0.360 train acc 0.878, test acc 0.722
epoch9, loss 0.329 train acc 0.888, test acc 0.737
epoch10, loss 0.334 train acc 0.889, test acc 0.710
epoch11, loss 0.306 train acc 0.896, test acc 0.744
epoch12, loss 0.305 train acc 0.899, test acc 0.675
epoch13, loss 0.320 train acc 0.893, test acc 0.727
epoch14, loss 0.307 train acc 0.896, test acc 0.704
epoch15, loss 0.283 train acc 0.905, test acc 0.736
epoch16, loss 0.292 train acc 0.900, test acc 0.692
epoch17, loss 0.301 train acc 0.901, test acc 0.736
epoch18, loss 0.287 train acc 0.904, test acc 0.716
epoch19, loss 0.308 train acc 0.898, test acc 0.725
epoch20, loss 0.289 t

# 三、小结测评

两个卷积的 padding=2 是怎么算出来的？不写 padding 的话，32×32 的图过完第一个卷积会变成多大？
首先我的 padding = 1 ，其次因为要输入输出相同，输入是 32，那么输出也得是所以根据公式来：（32-3+2p)/1 + 1 =32,则 p = 1，不写 padding 的话，那么则为 30 * 30

第一个 Linear(32*8*8, 64) 里的 32*8*8 是怎么来的？（提示：最后一次池化后，通道数×高×宽）
首先我的第一个 Linear 的参数是 64 * 4 * 4 ，其次经过 3 次的卷积 激活 池化，最终的输出为 64 通道、4 * 4 的尺寸

最后一层为什么是 Linear(64, 10)？换成 CIFAR-100 要改什么？、
把 10 变成100 

# 四、调参过程

第一版：严重过拟合<br>
- **结构**：16-32-64 卷积 + 512-256 全连接，无 Dropout，无 weight_decay<br>
- **结果**：训练 84%，测试峰值 67.5% 后一路跌到 56.5%，差距 28%<br>
- **诊断**：全连接头容量过大，模型背下训练集<br>
- **解决方向**：加正则

第二版：加了 Dropout + weight_decay<br>
- **结构**：全连接加 Dropout (0.5)，优化器加 weight_decay=5e-4<br>
- **结果**：训练冲到 100%，测试稳定 73%，差距 27%<br>
- **诊断**：正则阻止了测试集跳水，但模型容量还是太大，依然能背完训练集<br>
- **解决方向**：进一步降容量或加强正则

第三版：加强正则<br>
- **结构**：weight_decay 调到 1e-3，Dropout 0.5<br>
- **结果**：训练 71.5%，测试 69%，差距仅 2.5%<br>
- **诊断**：**过拟合解决了，但变成欠拟合**—— 正则太重，模型学不动<br>
- **解决方向**：回调正则力度，加学习率调度

第四版：回调正则
- **改动**：weight_decay 回调到 5e-4
- **结果**：训练 74%，测试 67%，还是不够
- **诊断**：检查代码发现两个遗漏 ——Dropout 只加在最后一层，512→256 之间没加

最终版：数据增强 + 余弦退火<br>
**改动**：RandomCrop(padding=4) + RandomHorizontalFlip（仅 train 用）；CosineAnnealingLR(T_max=30)；epoch 30；dropout/weight_decay 原样保留<br>
**结果**：train 0.904 / test 0.728，达到 >70% 目标（gap ≈ 17 点）<br>
**诊断**：增强 + lr 调度 + 更长训练是推过 70% 的关键；当前瓶颈是容量（3 层卷积泛化天花板 ≈73%），想再上探走 BatchNorm/加深加宽，不是继续加正则化

关键教训<br>
1. 先诊断再开药：gap 大（train 1.00 / test 0.60）→ 加正则化；两线都低（0.72 / 0.67）→ 加数据、加调度、加 epoch。两种病先后都得，药不能混吃<br>
2. 数据增强是 CIFAR-10 这类小数据集的默认配置，不是可选优化<br>
3. test acc ±1% 波动是评估噪声（测试集 1 万张，标准误 ≈0.44%），看趋势别看单 epoch